# Strict spatial training

v1 setup:

- H3 cell rows
- blocked spatial folds; preprocessing and models fit on train cells only
- RWI and accessibility excluded
- DHS used only as downstream labels
- train and test GNN graphs kept separate
- LOCO runs with the all-country setting

5 km DHS boundary buffer. refs: AlphaEarth, S2Vec and blocked spatial validation.

In [4]:
import importlib
import subprocess
import sys


def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        print(f"Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


for import_name, pip_name in [
    ("h3", "h3"),
    ("pyarrow", "pyarrow"),
    ("sklearn", "scikit-learn"),
    ("xgboost", "xgboost"),
]:
    ensure_package(import_name, pip_name)

In [5]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import math
import random
import warnings

import h3
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import RobustScaler

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    print(f"[WARN] xgboost unavailable; falling back to HistGradientBoostingRegressor: {exc}")

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except Exception as exc:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    print(f"[WARN] torch unavailable; SSL/GNN cells will be skipped: {exc}")

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=FutureWarning)

if IN_COLAB:
    drive.mount(str(Path("drive").resolve()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
# run config

SUBSET_NGA_TZA_ONLY = False
COUNTRY_SUBSET = ["NGA", "TZA"]

RUN_XGBOOST = True
RUN_BLOCK_ABLATIONS = False
RUN_SSL_EMBEDDING = True
RUN_GNN = True
RUN_LOCO = not SUBSET_NGA_TZA_ONLY

DROP_ACCESSIBILITY_FEATURES = True   # travel time can dominate the built-form test
INCLUDE_COORDINATES_AS_FEATURES = False   # no country/location shortcut
USE_GPU_ACCELERATION = True

# paths
PROJECT_ROOT = Path("drive") / "MyDrive" if IN_COLAB else Path(".")
OUTPUT_ROOT = PROJECT_ROOT / "h3_tabular_v2"
FINAL_DATA_PATH = OUTPUT_ROOT / "cells_full_tabular_v2.parquet"

RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
TRAINING_OUT = OUTPUT_ROOT / "training_outputs" / f"strict_spatial_{RUN_TAG}"
TRAINING_OUT.mkdir(parents=True, exist_ok=True)

# split
N_SPATIAL_FOLDS = 5
BLOCK_DEGREE_CANDIDATES = [1.0]  # fixed v1 block size
DHS_DISPLACEMENT_BUFFER_KM = 5.0  # cover standard rural displacement
MIN_LABELS_PER_FOLD = 30  # minimum workable outcome fold
RANDOM_STATE = 42

OUTCOME_COLS = [
    "dhs_wealth",
    "dhs_sanitation",
    "dhs_women_edu",
    "dhs_stunting",
    "dhs_mobile",
]

# first-pass model values
XGB_PARAMS = dict(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    tree_method="hist",
    eval_metric="rmse",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

SSL_Z_DIM = 256
SSL_HIDDEN = 512
SSL_DEPTH = 3
SSL_DROPOUT = 0.10
SSL_EPOCHS = 60
SSL_BATCH_SIZE = 16384
SSL_LR = 1e-3
SSL_MASK_RATE = 0.25
SSL_RECON_WEIGHT = 1.0
SSL_ALIGN_WEIGHT = 0.20
SSL_VAR_WEIGHT = 0.02

GNN_FEATURE_SETS_TO_RUN = ["ae_only", "full"]
GNN_HIDDEN = 128
GNN_LAYERS = 3
GNN_DROPOUT = 0.3
GNN_EPOCHS = 200
GNN_LR = 1e-3
GNN_WEIGHT_DECAY = 1e-4
GNN_K_RING = 1
GNN_SINGLE_TASK = True     # one model per outcome
GNN_VAL_FRAC = 0.2         # block-level validation
GNN_PATIENCE = 15

print(f"Output folder: {TRAINING_OUT}")

Output folder: h3_tabular_v2/training_outputs/strict_spatial_20260616_013629


## Load table

Input: `h3_tabular_v2/cells_full_tabular_v2.parquet`.

In [35]:
if not FINAL_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing final dataset: {FINAL_DATA_PATH}\n"
        "Run/copy the GEE outputs, then run the local WorldPop/OSM/DHS notebook final merge."
    )

cells = pd.read_parquet(FINAL_DATA_PATH)
print(f"Loaded {len(cells):,} rows and {cells.shape[1]:,} columns from {FINAL_DATA_PATH}")

required = {"country_iso3", "h3_index", "centroid_lat", "centroid_lon"}
missing_required = sorted(required - set(cells.columns))
if missing_required:
    raise KeyError(f"Input table missing required columns: {missing_required}")

rwi_cols = [
    c for c in cells.columns
    if c.lower().startswith("rwi") or "relative_wealth" in c.lower() or c.lower() == "has_rwi"
]
if rwi_cols:
    print(f"Dropping RWI/meta wealth columns completely: {rwi_cols}")
    cells = cells.drop(columns=rwi_cols)

if SUBSET_NGA_TZA_ONLY:
    before = len(cells)
    cells = cells[cells["country_iso3"].isin(COUNTRY_SUBSET)].copy()
    print(f"Subset to {COUNTRY_SUBSET}: {before:,} -> {len(cells):,} rows")

available_outcomes = [c for c in OUTCOME_COLS if c in cells.columns]
if not available_outcomes:
    raise KeyError(f"None of the requested DHS outcome columns are present: {OUTCOME_COLS}")

print("Countries:")
display(cells.groupby("country_iso3").size().rename("n_cells").to_frame())

label_summary = []
for col in available_outcomes:
    label_summary.append({
        "outcome": col,
        "n_labeled_cells": int(cells[col].notna().sum()),
        "mean": float(cells[col].mean(skipna=True)),
        "std": float(cells[col].std(skipna=True)),
    })
display(pd.DataFrame(label_summary))

Loaded 657,863 rows and 145 columns from h3_tabular_v2/cells_full_tabular_v2.parquet
Dropping RWI/meta wealth columns completely: ['has_rwi', 'rwi_mean', 'rwi_n_points', 'rwi_error_mean']
Countries:


,n_cells
country_iso3,
GHA,60433
GMB,2316
KEN,106744
NGA,190814
SLE,17349
TZA,158287
ZMB,121920


,outcome,n_labeled_cells,mean,std
0,dhs_wealth,5186,-0.071246,0.834031
1,dhs_sanitation,5186,0.587294,0.341456
2,dhs_women_edu,5186,7.394295,3.563986
3,dhs_stunting,5132,0.247230,0.221125
4,dhs_mobile,5186,0.873315,0.148327


## Spatial folds

Degree blocks with a 5 km boundary buffer. buffered DHS cells stay out of supervised fit and metrics; buffered unlabeled cells stay out of SSL fit.

In [36]:
def add_spatial_blocks(df: pd.DataFrame, block_deg: float) -> pd.DataFrame:
    out = df.copy()
    lon = out["centroid_lon"].astype(float)
    lat = out["centroid_lat"].astype(float)
    out["_gx"] = np.floor((lon + 180.0) / block_deg).astype("int32")
    out["_gy"] = np.floor((lat + 90.0) / block_deg).astype("int32")
    out["spatial_block"] = (
        out["country_iso3"].astype(str) + "_" +
        out["_gx"].astype(str) + "_" +
        out["_gy"].astype(str)
    )
    out["dist_to_block_boundary_km"] = distance_to_grid_boundary_km(lat, lon, block_deg)
    out["is_dhs_boundary_buffer"] = out["dist_to_block_boundary_km"] < DHS_DISPLACEMENT_BUFFER_KM
    return out


def distance_to_grid_boundary_km(lat: pd.Series, lon: pd.Series, block_deg: float) -> np.ndarray:
    lat = lat.to_numpy(dtype="float64")
    lon = lon.to_numpy(dtype="float64")
    x = (lon + 180.0) / block_deg
    y = (lat + 90.0) / block_deg
    frac_x = x - np.floor(x)
    frac_y = y - np.floor(y)
    dist_lon_deg = np.minimum(frac_x, 1.0 - frac_x) * block_deg
    dist_lat_deg = np.minimum(frac_y, 1.0 - frac_y) * block_deg
    km_per_lon_deg = 111.32 * np.maximum(np.cos(np.deg2rad(lat)), 0.05)
    dist_lon_km = dist_lon_deg * km_per_lon_deg
    dist_lat_km = dist_lat_deg * 111.32
    return np.minimum(dist_lon_km, dist_lat_km).astype("float32")


def assign_block_folds(df: pd.DataFrame, outcomes: list[str], n_folds: int, random_state: int) -> pd.Series:
    tmp = df[["spatial_block", "country_iso3", "h3_index"]].copy()
    tmp["label_any"] = df[outcomes].notna().any(axis=1)
    stats = (
        tmp.groupby("spatial_block", as_index=False)
        .agg(
            country_iso3=("country_iso3", "first"),
            n_cells=("h3_index", "size"),
            n_label_any=("label_any", "sum"),
        )
    )
    stats = stats.sample(frac=1.0, random_state=random_state)
    stats = stats.sort_values(["n_label_any", "n_cells"], ascending=False).reset_index(drop=True)

    fold_label_load = np.zeros(n_folds, dtype="float64")
    fold_cell_load = np.zeros(n_folds, dtype="float64")
    fold_country_load = [Counter() for _ in range(n_folds)]
    assignment = {}

    for row in stats.itertuples(index=False):
        scores = []
        for fold in range(n_folds):
            country_penalty = fold_country_load[fold][row.country_iso3]
            # balance labels first, then country spread, then cells
            scores.append((fold_label_load[fold], country_penalty, fold_cell_load[fold], fold))
        fold = min(scores)[-1]
        assignment[row.spatial_block] = fold
        fold_label_load[fold] += row.n_label_any
        fold_cell_load[fold] += row.n_cells
        fold_country_load[fold][row.country_iso3] += 1

    return df["spatial_block"].map(assignment).astype("int16")


def fold_outcome_counts(df: pd.DataFrame, outcomes: list[str]) -> pd.DataFrame:
    rows = []
    nonbuffer = ~df["is_dhs_boundary_buffer"].fillna(False)
    for fold in sorted(df["fold"].dropna().unique()):
        sub = df[df["fold"] == fold]
        sub_nb = sub[nonbuffer.loc[sub.index]]
        row = {
            "fold": int(fold),
            "n_cells": len(sub),
            "n_nonbuffer_cells": len(sub_nb),
            "n_buffer_cells": int(sub["is_dhs_boundary_buffer"].sum()),
            "buffer_fraction": float(sub["is_dhs_boundary_buffer"].mean()),
        }
        for outcome in outcomes:
            row[f"{outcome}_n"] = int(sub_nb[outcome].notna().sum())
            row[f"{outcome}_mean"] = float(sub_nb[outcome].mean(skipna=True))
        rows.append(row)
    return pd.DataFrame(rows)


def choose_spatial_folds(df: pd.DataFrame, outcomes: list[str]) -> tuple[pd.DataFrame, list[str], pd.DataFrame]:
    best = None
    for block_deg in BLOCK_DEGREE_CANDIDATES:
        cand = add_spatial_blocks(df, block_deg)
        cand["fold"] = assign_block_folds(cand, outcomes, N_SPATIAL_FOLDS, RANDOM_STATE)
        nonbuffer = ~cand["is_dhs_boundary_buffer"].fillna(False)
        # require enough labels to plausibly fill every fold
        active = [
            outcome for outcome in outcomes
            if int(cand.loc[nonbuffer, outcome].notna().sum()) >= MIN_LABELS_PER_FOLD * N_SPATIAL_FOLDS
        ]
        counts = fold_outcome_counts(cand, outcomes)
        min_count = 0
        if active:
            min_count = min(int(counts[f"{outcome}_n"].min()) for outcome in active)
        record = (min_count, block_deg, cand, active, counts)
        if best is None or record[0] > best[0]:
            best = record
        if active and min_count >= MIN_LABELS_PER_FOLD:
            print(f"Selected block_deg={block_deg} because all active outcomes clear the per-fold count gate.")
            return cand, active, counts

    assert best is not None
    min_count, block_deg, cand, active, counts = best
    print(
        f"[WARN] No block size cleared the per-fold count gate for every active outcome. "
        f"Using best available block_deg={block_deg}; best minimum count={min_count}."
    )
    if not active:
        active = [outcome for outcome in outcomes if cand[outcome].notna().sum() > 0]
        print(f"[WARN] Outcomes do not meet total label threshold. Keeping present outcomes for diagnostics: {active}")
    return cand, active, counts


cells, active_outcomes, fold_counts = choose_spatial_folds(cells, available_outcomes)
print(f"Active outcomes for model evaluation: {active_outcomes}")
display(fold_counts)

high_buffer = fold_counts[fold_counts["buffer_fraction"] > 0.20]
if len(high_buffer):
    print("[WARN] More than 20% of cells are inside the DHS displacement boundary buffer in these folds:")
    display(high_buffer[["fold", "n_cells", "n_buffer_cells", "buffer_fraction"]])
    print(
        "This is safe against leakage because the buffer is defined from the folding grid, "
        "but it may cost a meaningful amount of data. Consider coarser blocks, a smaller conservative buffer, "
        "or reporting the sensitivity."
    )
else:
    print("Buffered fraction check passed: every fold has <=20% buffered cells.")

fold_path = TRAINING_OUT / "spatial_fold_assignments.parquet"
cells[[
    "country_iso3", "h3_index", "centroid_lat", "centroid_lon",
    "spatial_block", "_gx", "_gy", "fold",
    "dist_to_block_boundary_km", "is_dhs_boundary_buffer",
]].to_parquet(fold_path, compression="snappy", index=False)
print(f"Wrote fold assignments -> {fold_path}")

Selected block_deg=1.0 because all active outcomes clear the per-fold count gate.
Active outcomes for model evaluation: ['dhs_wealth', 'dhs_sanitation', 'dhs_women_edu', 'dhs_stunting', 'dhs_mobile']


,fold,n_cells,n_nonbuffer_cells,n_buffer_cells,buffer_fraction,dhs_wealth_n,dhs_wealth_mean,dhs_sanitation_n,dhs_sanitation_mean,dhs_women_edu_n,dhs_women_edu_mean,dhs_stunting_n,dhs_stunting_mean,dhs_mobile_n,dhs_mobile_mean
0,0,135759,112856,22903,0.168703,852,-0.079915,852,0.613294,852,7.658507,845,0.231596,852,0.885150
1,1,123559,102402,21157,0.171230,836,-0.169625,836,0.551885,836,7.314126,829,0.246265,836,0.857188
2,2,132434,109611,22823,0.172335,861,-0.078903,861,0.581997,861,6.879078,850,0.260101,861,0.853823
3,3,133605,110497,23108,0.172958,863,-0.049769,863,0.569990,863,7.441377,857,0.236024,863,0.883793
4,4,132506,109226,23280,0.175690,854,0.041360,854,0.611870,854,7.731675,838,0.260475,854,0.881040


Buffered fraction check passed: every fold has <=20% buffered cells.
Wrote fold assignments -> h3_tabular_v2/training_outputs/strict_spatial_20260615_234216/spatial_fold_assignments.parquet


In [37]:
print("Fold country balance:")
display(pd.crosstab(cells["fold"], cells["country_iso3"]))

if "dhs_urban_rural" in cells.columns:
    print("DHS urban/rural distribution by fold, non-buffer labeled cells:")
    tmp = cells[(~cells["is_dhs_boundary_buffer"]) & cells[active_outcomes].notna().any(axis=1)].copy()
    display(pd.crosstab(tmp["fold"], tmp["dhs_urban_rural"], dropna=False))

diagnostic_cols = [
    c for c in ["pop_density", "pop_count", "ntl_mean_24m", "ndvi_mean", "road_density_km_per_km2"]
    if c in cells.columns
]
if diagnostic_cols:
    print("Fold covariate balance checks, median values:")
    display(cells.groupby("fold")[diagnostic_cols].median(numeric_only=True).round(4))

Fold country balance:


country_iso3,GHA,GMB,KEN,NGA,SLE,TZA,ZMB
fold,,,,,,,
0,15408,0,20819,42409,514,32061,24548
1,13807,100,21540,33557,4055,26580,23920
2,8851,766,22943,37584,4708,32548,25034
3,8169,1450,21339,41551,3408,35594,22094
4,14198,0,20103,35713,4664,31504,26324


DHS urban/rural distribution by fold, non-buffer labeled cells:


dhs_urban_rural,R,U
fold,,
0,520,332
1,540,296
2,536,325
3,507,356
4,484,370


Fold covariate balance checks, median values:


,pop_density,pop_count,ntl_mean_24m,ndvi_mean,road_density_km_per_km2
fold,,,,,
0,9.5415,48.972900,0.3327,0.5151,0.2300
1,12.3603,63.633202,0.3340,0.5178,0.1373
2,8.4958,44.960300,0.3331,0.5157,0.1380
3,10.7335,55.569698,0.3383,0.5475,0.1209
4,11.0043,57.448002,0.3352,0.5086,0.1403


## Features and preprocessing

RWI, DHS metadata, coordinates, quality flags and the road composite stay out of X. scaling and missingness handling fit on train cells only.

In [38]:
# feature selection
# true keeps demographics, temporal signals and morphology/topology only
ORTHOGONAL_ONLY = False

ID_AND_META_COLS = {
    "country_iso3", "h3_index", "geometry", "admin1_name",
    "spatial_block", "_gx", "_gy", "fold",
    "dist_to_block_boundary_km", "is_dhs_boundary_buffer",
    "has_dhs", "dhs_cluster_count", "dhs_cluster_id", "dhs_urban_rural", "dhs_admin1_name",
}

if not INCLUDE_COORDINATES_AS_FEATURES:
    ID_AND_META_COLS |= {"centroid_lat", "centroid_lon"}

# remove duplicate timing, latitude proxy and hand-built composite
DROP_EXACT_COLS = {
    "road_grid_like_index",
    "area_km2",
    "ndvi_peak_month",
    "ndvi_peak_period",
    "road_dead_end_count",
}

# quality flags can leak cloud/coverage geography
QUALITY_FLAG_SUFFIXES = ("_reliable",)
QUALITY_FLAG_EXACT = {
    "ndvi_valid_periods",
    "ndvi_valid_months",
    "ntl_valid_months",
}

ACCESSIBILITY_PATTERNS = [
    "accessibility", "travel_time", "ttc", "tth", "market_access",
]


def is_quality_flag(col: str) -> bool:
    low = col.lower()
    if col in QUALITY_FLAG_EXACT:
        return True
    if low.endswith(QUALITY_FLAG_SUFFIXES):
        return True
    if low.startswith("valid_") or low.endswith("_valid"):
        return True
    return False


def is_forbidden_feature(col: str) -> bool:
    low = col.lower()
    if col in ID_AND_META_COLS or col in DROP_EXACT_COLS:
        return True
    if col in OUTCOME_COLS:
        return True
    if low.startswith("dhs_"):
        return True
    if low.startswith("rwi") or "relative_wealth" in low or low == "has_rwi":
        return True
    if low.endswith("_effective_year") or low in {"reference_year", "survey_year"}:
        return True
    if low.endswith("_source") or low.endswith("_effective_source"):
        return True
    if is_quality_flag(col):
        return True
    if DROP_ACCESSIBILITY_FEATURES and any(pat in low for pat in ACCESSIBILITY_PATTERNS):
        return True
    return False


def select_feature_columns(df: pd.DataFrame) -> list[str]:
    cols = []
    for col in df.columns:
        if is_forbidden_feature(col):
            continue
        s = df[col]
        if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
            cols.append(col)
    return cols


# amount vs structure; GHSL height stays with structure

BUILDING_STRUCTURE = {
    "gob_building_size_gini",
    "gob_nn_cv",
    "gob_spacing_regularity",
    "gob_orientation_entropy",
    "ghsl_built_height_mean_m",
    "ghsl_built_height_p90_m",
    "ghsl_built_height_max_m",
}
ROAD_TOPOLOGY = {
    "road_dead_end_ratio",
    "road_beta_connectivity",
    "road_bearing_entropy",
    "road_degree3_share",
    "road_degree4_share",
    "road_degree5plus_share",
    "road_degree_mean",
}
NTL_TEMPORAL = {
    "ntl_log_slope_24m",
    "ntl_log_volatility_24m",
    "ntl_raw_volatility_24m",
}
NDVI_TEMPORAL = {
    "ndvi_seasonal_amplitude",
    "ndvi_peak_doy",
}

# fields less directly visible in AE
ORTHOGONAL_BLOCKS = {
    "demographics",
    "nightlights_temporal",
    "ndvi_temporal",
    "buildings_structure",
    "roads_topology",
}


def feature_group_columns(cols: list[str]) -> dict[str, list[str]]:
    groups = {
        "demographics": [],
        "nightlights_temporal": [],
        "nightlights_amount": [],
        "ndvi_temporal": [],
        "ndvi_amount": [],
        "buildings_structure": [],
        "buildings_amount": [],
        "roads_topology": [],
        "roads_amount": [],
        "other_tabular": [],
    }
    for c in cols:
        low = c.lower()
        if c.startswith("ae_A"):
            continue
        if low.startswith("pop_") or "dep_ratio" in low or "sex_ratio" in low:
            groups["demographics"].append(c)
        elif low.startswith("ntl_"):
            groups["nightlights_temporal" if c in NTL_TEMPORAL else "nightlights_amount"].append(c)
        elif low.startswith("ndvi_"):
            groups["ndvi_temporal" if c in NDVI_TEMPORAL else "ndvi_amount"].append(c)
        elif low.startswith("gob_") or low.startswith("ghsl_"):
            groups["buildings_structure" if c in BUILDING_STRUCTURE else "buildings_amount"].append(c)
        elif low.startswith("road_"):
            groups["roads_topology" if c in ROAD_TOPOLOGY else "roads_amount"].append(c)
        else:
            groups["other_tabular"].append(c)
    return groups


# build feature lists

all_feature_cols = select_feature_columns(cells)
ae_cols = [c for c in all_feature_cols if c.startswith("ae_A")]
all_tab_cols = [c for c in all_feature_cols if c not in ae_cols]

feature_groups = feature_group_columns(all_feature_cols)

if ORTHOGONAL_ONLY:
    tab_cols = [c for blk in ORTHOGONAL_BLOCKS for c in feature_groups.get(blk, [])]
    print(f"[ORTHOGONAL_ONLY] Restricting to {len(tab_cols)} orthogonal tabular features.")
else:
    tab_cols = all_tab_cols

feature_cols = ae_cols + tab_cols

print(f"Selected {len(feature_cols):,} features: {len(ae_cols)} AlphaEarth, {len(tab_cols)} tabular "
      f"(ORTHOGONAL_ONLY={ORTHOGONAL_ONLY}).")

if len(ae_cols) != 64:
    print(f"[WARN] Expected 64 AlphaEarth columns, found {len(ae_cols)}.")

print("\nTabular feature groups (after drops):")
for name, cols in feature_groups.items():
    tag = " [orthogonal]" if name in ORTHOGONAL_BLOCKS else ""
    if cols:
        print(f"  {name}{tag}: {len(cols)}  ->  {sorted(cols)}")

if feature_groups["other_tabular"]:
    print(f"\n[CHECK] other_tabular catch-all: {sorted(feature_groups['other_tabular'])}")
else:
    print("\n[OK] other_tabular is empty.")


# five feature sets; ortho + amount partition tab

def build_feature_sets():
    ortho = [c for blk in ORTHOGONAL_BLOCKS for c in feature_groups.get(blk, [])]
    amount = [c for blk in feature_groups
              if blk not in ORTHOGONAL_BLOCKS and blk != "other_tabular"
              for c in feature_groups[blk]]
    sets = {"ae_only": ae_cols}
    if tab_cols:               sets["tab_only"] = tab_cols
    if ae_cols and tab_cols:   sets["full"] = ae_cols + tab_cols
    if ae_cols and ortho:      sets["ae_plus_orthogonal"] = ae_cols + ortho
    if ae_cols and amount:     sets["ae_plus_amount"] = ae_cols + amount
    return sets


feature_sets = build_feature_sets()
print("\nFeature sets:")
for name, cols in feature_sets.items():
    print(f"  {name}: {len(cols)}")

# partition sanity
_ortho = {c for blk in ORTHOGONAL_BLOCKS for c in feature_groups.get(blk, [])}
_amount = {c for blk in feature_groups
           if blk not in ORTHOGONAL_BLOCKS and blk != "other_tabular"
           for c in feature_groups[blk]}
if _ortho | _amount != set(all_tab_cols) or (_ortho & _amount):
    print(f"[WARN] ortho/amount do not cleanly partition tab. "
          f"ortho={len(_ortho)}, amount={len(_amount)}, tab={len(all_tab_cols)}, "
          f"overlap={len(_ortho & _amount)}, "
          f"missing={sorted(set(all_tab_cols) - _ortho - _amount)}")
else:
    print(f"[OK] ortho({len(_ortho)}) + amount({len(_amount)}) cleanly partition tab({len(all_tab_cols)}).")


feature_sets = build_feature_sets()
print("\nFeature sets:")
for name, cols in feature_sets.items():
    print(f"  {name}: {len(cols)}")

Selected 101 features: 64 AlphaEarth, 37 tabular (ORTHOGONAL_ONLY=False).

Tabular feature groups (after drops):
  demographics [orthogonal]: 5  ->  ['elderly_dep_ratio', 'pop_count', 'pop_density', 'sex_ratio', 'youth_dep_ratio']
  nightlights_temporal [orthogonal]: 3  ->  ['ntl_log_slope_24m', 'ntl_log_volatility_24m', 'ntl_raw_volatility_24m']
  nightlights_amount: 1  ->  ['ntl_mean_24m']
  ndvi_temporal [orthogonal]: 2  ->  ['ndvi_peak_doy', 'ndvi_seasonal_amplitude']
  ndvi_amount: 1  ->  ['ndvi_mean']
  buildings_structure [orthogonal]: 7  ->  ['ghsl_built_height_max_m', 'ghsl_built_height_mean_m', 'ghsl_built_height_p90_m', 'gob_building_size_gini', 'gob_nn_cv', 'gob_orientation_entropy', 'gob_spacing_regularity']
  buildings_amount: 6  ->  ['gob_building_area_mean_m2', 'gob_building_area_std_m2', 'gob_building_area_total_m2', 'gob_building_count', 'gob_building_density_per_km2', 'gob_footprint_fraction']
  roads_topology [orthogonal]: 7  ->  ['road_bearing_entropy', 'road_beta_

In [39]:
@dataclass
class MatrixSpec:
    columns: list[str]
    missing_flag_columns: list[str]
    scaler: RobustScaler


def _numeric_frame(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    x = df[cols].copy()
    for col in cols:
        if pd.api.types.is_bool_dtype(x[col]):
            x[col] = x[col].astype("float32")
        else:
            x[col] = pd.to_numeric(x[col], errors="coerce")
    return x


def fit_matrix(df: pd.DataFrame, row_mask: pd.Series | np.ndarray, cols: list[str]) -> tuple[np.ndarray, MatrixSpec]:
    x = _numeric_frame(df.loc[row_mask], cols)
    missing_base = [c for c in cols if x[c].isna().any()]  # keep train-side missingness signal
    for c in missing_base:
        x[f"miss__{c}"] = x[c].isna().astype("float32")
    x = x.fillna(0.0).astype("float32")  # zero only after missing flags
    scaler = RobustScaler()  # heavy-tailed spatial features
    arr = scaler.fit_transform(x).astype("float32")
    return arr, MatrixSpec(columns=list(x.columns), missing_flag_columns=[f"miss__{c}" for c in missing_base], scaler=scaler)


def transform_matrix(df: pd.DataFrame, row_mask: pd.Series | np.ndarray, cols: list[str], spec: MatrixSpec) -> np.ndarray:
    x = _numeric_frame(df.loc[row_mask], cols)
    for miss_col in spec.missing_flag_columns:
        base = miss_col.replace("miss__", "", 1)
        x[miss_col] = x[base].isna().astype("float32") if base in x.columns else 0.0
    x = x.fillna(0.0).astype("float32")
    x = x.reindex(columns=spec.columns, fill_value=0.0)
    return spec.scaler.transform(x).astype("float32")


def get_fold_masks(df: pd.DataFrame, fold: int, outcome: str | None = None, use_buffer: bool = True) -> tuple[pd.Series, pd.Series]:
    nonbuffer = ~df["is_dhs_boundary_buffer"].fillna(False) if use_buffer else pd.Series(True, index=df.index)
    train = (df["fold"] != fold) & nonbuffer
    test = (df["fold"] == fold) & nonbuffer
    if outcome is not None:
        train = train & df[outcome].notna()
        test = test & df[outcome].notna()
    return train, test


from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def regression_metrics(y_true, y_pred) -> dict[str, float | int]:
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")

    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[ok]
    y_pred = y_pred[ok]

    if len(y_true) == 0:
        return {"n": 0, "r2": np.nan, "rmse": np.nan, "mae": np.nan, "spearman": np.nan}

    r2 = r2_score(y_true, y_pred) if len(y_true) >= 2 and np.nanstd(y_true) > 0 else np.nan
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    spearman = (
        float(pd.Series(y_true).corr(pd.Series(y_pred), method="spearman"))
        if len(y_true) >= 3 else np.nan
    )

    return {
        "n": int(len(y_true)),
        "r2": float(r2),
        "rmse": rmse,
        "mae": mae,
        "spearman": spearman,
    }


def summarize_metrics(metrics: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    metric_cols = ["r2", "rmse", "mae", "spearman", "n"]
    summary = (
        metrics.groupby(group_cols)[metric_cols]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )
    summary.columns = ["_".join([p for p in col if p]) for col in summary.columns.to_flat_index()]
    return summary


def pooled_metrics_from_predictions(predictions: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []
    if predictions is None or len(predictions) == 0:
        return pd.DataFrame()
    for key, g in predictions.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        row.update(regression_metrics(g["y_true"].to_numpy(), g["y_pred"].to_numpy()))
        if "fold" in g.columns:
            row["n_folds"] = int(g["fold"].nunique())
        if "holdout_country" in g.columns:
            row["n_holdout_countries"] = int(g["holdout_country"].nunique())
        rows.append(row)
    return pd.DataFrame(rows)

## XGBoost

Fold-safe downstream probes for each feature set.

In [40]:
import warnings
warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)

In [41]:
def make_xgb_or_fallback():
    if xgb is not None:
        params = dict(XGB_PARAMS)
        if USE_GPU_ACCELERATION and torch is not None and torch.cuda.is_available():
            # xgboost 2.x GPU path
            params["device"] = "cuda"
        return xgb.XGBRegressor(**params)
    return HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.04,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )


def run_xgb_spatial_cv(df: pd.DataFrame, outcomes: list[str], feature_sets: dict[str, list[str]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows = []
    pred_frames = []
    for outcome in outcomes:
        for set_name, cols in feature_sets.items():
            print(f"\n=== XGB spatial CV: {outcome} | {set_name} | {len(cols)} features ===")
            for fold in range(N_SPATIAL_FOLDS):
                train_mask, test_mask = get_fold_masks(df, fold, outcome=outcome, use_buffer=True)  # buffer out of fit + metrics
                n_train = int(train_mask.sum())
                n_test = int(test_mask.sum())
                if n_train < 20 or n_test < 2:
                    print(f"[SKIP] fold={fold}: train={n_train}, test={n_test}")
                    continue
                x_train, spec = fit_matrix(df, train_mask, cols)
                x_test = transform_matrix(df, test_mask, cols, spec)
                y_train = df.loc[train_mask, outcome].astype("float32").to_numpy()
                y_test = df.loc[test_mask, outcome].astype("float32").to_numpy()

                model = make_xgb_or_fallback()
                try:
                    model.fit(x_train, y_train)
                except Exception as exc:
                    if xgb is not None and "cuda" in str(exc).lower():
                        print(f"[WARN] CUDA XGBoost failed on fold={fold}; retrying CPU hist. Error: {exc}")
                        params = dict(XGB_PARAMS)
                        model = xgb.XGBRegressor(**params)
                        model.fit(x_train, y_train)
                    else:
                        raise
                pred = model.predict(x_test)
                m = regression_metrics(y_test, pred)
                m.update({
                    "experiment": "spatial_cv",
                    "model": "xgb" if xgb is not None else "hist_gradient_boosting",
                    "outcome": outcome,
                    "feature_set": set_name,
                    "fold": fold,
                    "n_train": n_train,
                    "n_test": n_test,
                })
                metric_rows.append(m)
                print(f"fold={fold} n_train={n_train:,} n_test={n_test:,} r2={m['r2']:.3f} rmse={m['rmse']:.3f} spearman={m['spearman']:.3f}")

                pred_frames.append(pd.DataFrame({
                    "experiment": "spatial_cv",
                    "model": m["model"],
                    "outcome": outcome,
                    "feature_set": set_name,
                    "fold": fold,
                    "country_iso3": df.loc[test_mask, "country_iso3"].to_numpy(),
                    "h3_index": df.loc[test_mask, "h3_index"].to_numpy(),
                    "y_true": y_test,
                    "y_pred": pred.astype("float32"),
                }))

    metrics = pd.DataFrame(metric_rows)
    predictions = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    return metrics, predictions


if RUN_XGBOOST:
    xgb_metrics, xgb_predictions = run_xgb_spatial_cv(cells, active_outcomes, feature_sets)
    xgb_metrics_path = TRAINING_OUT / "xgb_spatial_cv_metrics.csv"
    xgb_predictions_path = TRAINING_OUT / "xgb_spatial_cv_predictions.parquet"
    xgb_metrics.to_csv(xgb_metrics_path, index=False)
    xgb_predictions.to_parquet(xgb_predictions_path, compression="snappy", index=False)
    print(f"\nWrote {xgb_metrics_path}")
    print(f"Wrote {xgb_predictions_path}")
    display(summarize_metrics(xgb_metrics, ["experiment", "model", "outcome", "feature_set"]).round(4))
else:
    xgb_metrics = pd.DataFrame()
    xgb_predictions = pd.DataFrame()
    print("RUN_XGBOOST=False")


=== XGB spatial CV: dhs_wealth | ae_only | 64 features ===
fold=0 n_train=3,414 n_test=852 r2=0.585 rmse=0.530 spearman=0.766
fold=1 n_train=3,430 n_test=836 r2=0.587 rmse=0.493 spearman=0.750
fold=2 n_train=3,405 n_test=861 r2=0.650 rmse=0.487 spearman=0.805
fold=3 n_train=3,403 n_test=863 r2=0.640 rmse=0.498 spearman=0.798
fold=4 n_train=3,412 n_test=854 r2=0.682 rmse=0.524 spearman=0.818

=== XGB spatial CV: dhs_wealth | tab_only | 37 features ===
fold=0 n_train=3,414 n_test=852 r2=0.705 rmse=0.447 spearman=0.844
fold=1 n_train=3,430 n_test=836 r2=0.704 rmse=0.418 spearman=0.845
fold=2 n_train=3,405 n_test=861 r2=0.757 rmse=0.405 spearman=0.867
fold=3 n_train=3,403 n_test=863 r2=0.740 rmse=0.424 spearman=0.867
fold=4 n_train=3,412 n_test=854 r2=0.756 rmse=0.459 spearman=0.875

=== XGB spatial CV: dhs_wealth | full | 101 features ===
fold=0 n_train=3,414 n_test=852 r2=0.715 rmse=0.439 spearman=0.849
fold=1 n_train=3,430 n_test=836 r2=0.712 rmse=0.412 spearman=0.844
fold=2 n_train=3,

,experiment,model,outcome,feature_set,r2_mean,r2_std,r2_min,r2_max,rmse_mean,rmse_std,rmse_min,rmse_max,mae_mean,mae_std,mae_min,mae_max,spearman_mean,spearman_std,spearman_min,spearman_max,n_mean,n_std,n_min,n_max
0,spatial_cv,xgb,dhs_mobile,ae_only,0.4572,0.0636,0.4001,0.5288,0.1095,0.0109,0.0961,0.1239,0.0761,0.0060,0.0682,0.0842,0.6217,0.0471,0.5432,0.6601,853.2,10.6630,836,863
1,spatial_cv,xgb,dhs_mobile,ae_plus_amount,0.4916,0.0791,0.4155,0.5842,0.1057,0.0100,0.0958,0.1205,0.0727,0.0064,0.0654,0.0820,0.6579,0.0427,0.5905,0.6939,853.2,10.6630,836,863
2,spatial_cv,xgb,dhs_mobile,ae_plus_orthogonal,0.4849,0.0726,0.4051,0.5643,0.1065,0.0095,0.0966,0.1192,0.0732,0.0057,0.0659,0.0800,0.6504,0.0478,0.5819,0.6953,853.2,10.6630,836,863
3,spatial_cv,xgb,dhs_mobile,full,0.5031,0.0810,0.4208,0.5917,0.1045,0.0101,0.0952,0.1195,0.0721,0.0063,0.0647,0.0808,0.6612,0.0413,0.6007,0.6982,853.2,10.6630,836,863
4,spatial_cv,xgb,dhs_mobile,tab_only,0.4851,0.0776,0.3660,0.5546,0.1063,0.0071,0.0997,0.1151,0.0740,0.0051,0.0687,0.0797,0.6464,0.0583,0.5682,0.6956,853.2,10.6630,836,863
5,spatial_cv,xgb,dhs_sanitation,ae_only,0.4180,0.0541,0.3508,0.4820,0.2586,0.0061,0.2483,0.2629,0.2046,0.0066,0.1959,0.2097,0.6549,0.0462,0.5983,0.7070,853.2,10.6630,836,863
6,spatial_cv,xgb,dhs_sanitation,ae_plus_amount,0.5095,0.0472,0.4527,0.5713,0.2374,0.0089,0.2216,0.2425,0.1845,0.0078,0.1714,0.1900,0.7200,0.0290,0.6860,0.7568,853.2,10.6630,836,863
7,spatial_cv,xgb,dhs_sanitation,ae_plus_orthogonal,0.4915,0.0472,0.4248,0.5410,0.2418,0.0077,0.2293,0.2485,0.1896,0.0079,0.1763,0.1959,0.7073,0.0296,0.6664,0.7401,853.2,10.6630,836,863
8,spatial_cv,xgb,dhs_sanitation,full,0.5216,0.0506,0.4551,0.5811,0.2344,0.0088,0.2190,0.2402,0.1828,0.0081,0.1690,0.1890,0.7271,0.0286,0.6914,0.7619,853.2,10.6630,836,863
9,spatial_cv,xgb,dhs_sanitation,tab_only,0.4990,0.0534,0.4262,0.5623,0.2399,0.0105,0.2239,0.2496,0.1894,0.0092,0.1738,0.1951,0.7143,0.0322,0.6728,0.7486,853.2,10.6630,836,863


## SSL embedding

Encoder fit on train-fold features without DHS labels. frozen Ridge probes use train-fold DHS labels.

In [42]:
if torch is not None:
    def make_mlp(in_dim: int, hidden: int, out_dim: int, depth: int, dropout: float) -> nn.Sequential:
        layers = []
        dim = in_dim
        for _ in range(max(depth - 1, 1)):
            layers.extend([
                nn.Linear(dim, hidden),
                nn.GELU(),
                nn.LayerNorm(hidden),
                nn.Dropout(dropout),
            ])
            dim = hidden
        layers.append(nn.Linear(dim, out_dim))
        return nn.Sequential(*layers)


    class FusedAutoencoder(nn.Module):
        def __init__(self, n_ae: int, n_tab: int, n_total: int, hidden: int, z_dim: int):
            super().__init__()
            self.n_ae = n_ae
            self.n_tab = n_tab
            self.ae_encoder = make_mlp(max(n_ae, 1), hidden, z_dim, SSL_DEPTH, SSL_DROPOUT)
            self.tab_encoder = make_mlp(max(n_tab, 1), hidden, z_dim, SSL_DEPTH, SSL_DROPOUT)
            self.fuse = make_mlp(z_dim * 2, hidden, z_dim, SSL_DEPTH, SSL_DROPOUT)
            self.decoder = make_mlp(z_dim, hidden, n_total, SSL_DEPTH, SSL_DROPOUT)

        def forward(self, x, ae_idx, tab_idx):
            if len(ae_idx) == 0:
                ae = torch.zeros((x.shape[0], 1), dtype=x.dtype, device=x.device)
            else:
                ae = x[:, ae_idx]
            if len(tab_idx) == 0:
                tab = torch.zeros((x.shape[0], 1), dtype=x.dtype, device=x.device)
            else:
                tab = x[:, tab_idx]
            z_ae = self.ae_encoder(ae)
            z_tab = self.tab_encoder(tab)
            z = self.fuse(torch.cat([z_ae, z_tab], dim=1))
            recon = self.decoder(z)
            return z, z_ae, z_tab, recon


def variance_loss(z: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    std = torch.sqrt(z.var(dim=0) + eps)
    return torch.mean(torch.relu(1.0 - std))


def train_ssl_encoder(x_train: np.ndarray, spec: MatrixSpec) -> tuple[FusedAutoencoder, list[int], list[int]]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ae_idx = [i for i, c in enumerate(spec.columns) if c.startswith("ae_A") or c.startswith("miss__ae_A")]
    tab_idx = [i for i in range(len(spec.columns)) if i not in set(ae_idx)]

    model = FusedAutoencoder(
        n_ae=len(ae_idx),
        n_tab=len(tab_idx),
        n_total=x_train.shape[1],
        hidden=SSL_HIDDEN,
        z_dim=SSL_Z_DIM,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=SSL_LR, weight_decay=1e-4)
    ds = TensorDataset(torch.from_numpy(x_train.astype("float32")))
    dl = DataLoader(ds, batch_size=SSL_BATCH_SIZE, shuffle=True, drop_last=False)

    model.train()
    for epoch in range(1, SSL_EPOCHS + 1):
        losses = []
        for (xb,) in dl:
            xb = xb.to(device)
            keep = (torch.rand_like(xb) > SSL_MASK_RATE).float()
            xm = xb * keep
            z, z_ae, z_tab, recon = model(xm, ae_idx, tab_idx)
            recon_loss = torch.mean((recon - xb) ** 2)
            align_loss = 1.0 - torch.cosine_similarity(z_ae, z_tab, dim=1).mean()
            var_loss = variance_loss(z)
            loss = SSL_RECON_WEIGHT * recon_loss + SSL_ALIGN_WEIGHT * align_loss + SSL_VAR_WEIGHT * var_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(float(loss.detach().cpu()))
        if epoch == 1 or epoch % 5 == 0 or epoch == SSL_EPOCHS:
            print(f"ssl epoch {epoch:03d}/{SSL_EPOCHS}: loss={np.mean(losses):.5f}")
    return model.eval(), ae_idx, tab_idx


def encode_with_ssl(model: FusedAutoencoder, x: np.ndarray, ae_idx: list[int], tab_idx: list[int]) -> np.ndarray:
    device = next(model.parameters()).device
    out = []
    dl = DataLoader(TensorDataset(torch.from_numpy(x.astype("float32"))), batch_size=8192, shuffle=False)
    with torch.no_grad():
        for (xb,) in dl:
            xb = xb.to(device)
            z, _, _, _ = model(xb, ae_idx, tab_idx)
            out.append(z.detach().cpu().numpy())
    return np.vstack(out).astype("float32")


def run_ssl_frozen_spatial_cv(df: pd.DataFrame, outcomes: list[str], full_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows = []
    pred_frames = []
    for fold in range(N_SPATIAL_FOLDS):
        ssl_train_mask = (df["fold"] != fold) & (~df["is_dhs_boundary_buffer"].fillna(False))  # held-out fold never enters encoder fit
        ssl_test_mask = (df["fold"] == fold) & (~df["is_dhs_boundary_buffer"].fillna(False))
        if int(ssl_train_mask.sum()) < 100 or int(ssl_test_mask.sum()) < 10:
            print(f"[SKIP SSL] fold={fold}: ssl_train={int(ssl_train_mask.sum())}, ssl_test={int(ssl_test_mask.sum())}")
            continue

        print(f"\n=== SSL fold {fold}: train cells={int(ssl_train_mask.sum()):,}, test cells={int(ssl_test_mask.sum()):,} ===")
        x_ssl_train, spec = fit_matrix(df, ssl_train_mask, full_cols)
        model, ae_idx, tab_idx = train_ssl_encoder(x_ssl_train, spec)

        for outcome in outcomes:
            train_label_mask, test_label_mask = get_fold_masks(df, fold, outcome=outcome, use_buffer=True)
            if int(train_label_mask.sum()) < 20 or int(test_label_mask.sum()) < 2:
                print(f"[SKIP SSL PROBE] fold={fold} outcome={outcome}: train={int(train_label_mask.sum())}, test={int(test_label_mask.sum())}")
                continue

            x_probe_train = transform_matrix(df, train_label_mask, full_cols, spec)
            x_probe_test = transform_matrix(df, test_label_mask, full_cols, spec)
            z_train = encode_with_ssl(model, x_probe_train, ae_idx, tab_idx)
            z_test = encode_with_ssl(model, x_probe_test, ae_idx, tab_idx)
            y_train = df.loc[train_label_mask, outcome].astype("float32").to_numpy()
            y_test = df.loc[test_label_mask, outcome].astype("float32").to_numpy()

            probe = RidgeCV(alphas=np.logspace(-3, 3, 13))  # frozen embedding, linear probe
            probe.fit(z_train, y_train)
            pred = probe.predict(z_test).astype("float32")
            m = regression_metrics(y_test, pred)
            m.update({
                "experiment": "spatial_cv",
                "model": "ssl_frozen_z_ridge",
                "outcome": outcome,
                "feature_set": "ssl_full_z",
                "fold": fold,
                "n_train": int(train_label_mask.sum()),
                "n_test": int(test_label_mask.sum()),
                "z_dim": SSL_Z_DIM,
            })
            metric_rows.append(m)
            print(f"fold={fold} {outcome}: r2={m['r2']:.3f} rmse={m['rmse']:.3f} spearman={m['spearman']:.3f}")
            pred_frames.append(pd.DataFrame({
                "experiment": "spatial_cv",
                "model": "ssl_frozen_z_ridge",
                "outcome": outcome,
                "feature_set": "ssl_full_z",
                "fold": fold,
                "country_iso3": df.loc[test_label_mask, "country_iso3"].to_numpy(),
                "h3_index": df.loc[test_label_mask, "h3_index"].to_numpy(),
                "y_true": y_test,
                "y_pred": pred,
            }))
    metrics = pd.DataFrame(metric_rows)
    predictions = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    return metrics, predictions


if RUN_SSL_EMBEDDING:
    if torch is None:
        print("[SKIP] torch unavailable.")
        ssl_metrics = pd.DataFrame()
        ssl_predictions = pd.DataFrame()
    elif "full" not in feature_sets:
        print("[SKIP] full feature set unavailable.")
        ssl_metrics = pd.DataFrame()
        ssl_predictions = pd.DataFrame()
    else:
        ssl_metrics, ssl_predictions = run_ssl_frozen_spatial_cv(cells, active_outcomes, feature_sets["full"])
        ssl_metrics_path = TRAINING_OUT / "ssl_frozen_spatial_cv_metrics.csv"
        ssl_predictions_path = TRAINING_OUT / "ssl_frozen_spatial_cv_predictions.parquet"
        ssl_metrics.to_csv(ssl_metrics_path, index=False)
        ssl_predictions.to_parquet(ssl_predictions_path, compression="snappy", index=False)
        print(f"\nWrote {ssl_metrics_path}")
        print(f"Wrote {ssl_predictions_path}")
        if len(ssl_metrics):
            display(summarize_metrics(ssl_metrics, ["experiment", "model", "outcome", "feature_set"]).round(4))
else:
    ssl_metrics = pd.DataFrame()
    ssl_predictions = pd.DataFrame()
    print("RUN_SSL_EMBEDDING=False")


=== SSL fold 0: train cells=431,736, test cells=112,856 ===
ssl epoch 001/60: loss=63.97790
ssl epoch 005/60: loss=53.74160
ssl epoch 010/60: loss=46.69987
ssl epoch 015/60: loss=45.81229
ssl epoch 020/60: loss=41.30735
ssl epoch 025/60: loss=49.32448
ssl epoch 030/60: loss=35.49497
ssl epoch 035/60: loss=34.87294
ssl epoch 040/60: loss=32.65206
ssl epoch 045/60: loss=33.22585
ssl epoch 050/60: loss=30.01234
ssl epoch 055/60: loss=30.59111
ssl epoch 060/60: loss=26.92021
fold=0 dhs_wealth: r2=0.658 rmse=0.481 spearman=0.818
fold=0 dhs_sanitation: r2=0.391 rmse=0.254 spearman=0.655
fold=0 dhs_women_edu: r2=0.549 rmse=2.318 spearman=0.726
fold=0 dhs_stunting: r2=0.157 rmse=0.206 spearman=0.362
fold=0 dhs_mobile: r2=0.263 rmse=0.107 spearman=0.518

=== SSL fold 1: train cells=442,190, test cells=102,402 ===
ssl epoch 001/60: loss=72.54261
ssl epoch 005/60: loss=61.52407
ssl epoch 010/60: loss=53.71604
ssl epoch 015/60: loss=48.97267
ssl epoch 020/60: loss=48.87549
ssl epoch 025/60: loss=

,experiment,model,outcome,feature_set,r2_mean,r2_std,r2_min,r2_max,rmse_mean,rmse_std,rmse_min,rmse_max,mae_mean,mae_std,mae_min,mae_max,spearman_mean,spearman_std,spearman_min,spearman_max,n_mean,n_std,n_min,n_max
0,spatial_cv,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,0.3148,0.1375,0.1045,0.4631,0.1232,0.0208,0.1075,0.1593,0.0845,0.0073,0.0765,0.0954,0.5959,0.0611,0.5180,0.6686,853.2,10.6630,836,863
1,spatial_cv,ssl_frozen_z_ridge,dhs_sanitation,ssl_full_z,0.4618,0.0485,0.3914,0.5104,0.2487,0.0069,0.2368,0.2537,0.1973,0.0051,0.1892,0.2024,0.6939,0.0275,0.6552,0.7237,853.2,10.6630,836,863
2,spatial_cv,ssl_frozen_z_ridge,dhs_stunting,ssl_full_z,0.1567,0.0391,0.1126,0.2196,0.2016,0.0076,0.1935,0.2109,0.1589,0.0060,0.1512,0.1654,0.3844,0.0379,0.3458,0.4457,843.8,10.8028,829,857
3,spatial_cv,ssl_frozen_z_ridge,dhs_wealth,ssl_full_z,0.6986,0.0560,0.6223,0.7482,0.4548,0.0270,0.4147,0.4809,0.3463,0.0224,0.3146,0.3669,0.8424,0.0255,0.8114,0.8656,853.2,10.6630,836,863
4,spatial_cv,ssl_frozen_z_ridge,dhs_women_edu,ssl_full_z,0.5385,0.1177,0.3397,0.6316,2.4004,0.2559,2.2418,2.8539,1.7862,0.0389,1.7425,1.8427,0.7592,0.0219,0.7258,0.7805,853.2,10.6630,836,863


## GNN

H3 adjacency inside separate train and test subgraphs. no held-out boundary edges.

In [49]:
def h3_grid_disk(cell: str, k: int) -> set[str]:
    if hasattr(h3, "grid_disk"):
        return set(h3.grid_disk(cell, k))
    return set(h3.k_ring(cell, k))


def build_edge_index(h3_values: np.ndarray, k: int = 1) -> np.ndarray:
    id_map = {str(h): i for i, h in enumerate(h3_values)}
    src, dst = [], []
    for h, i in id_map.items():
        for nb in h3_grid_disk(h, k):
            nb = str(nb)
            if nb == h:
                continue
            j = id_map.get(nb)
            if j is not None:
                src.append(j)
                dst.append(i)
    if not src:
        return np.zeros((2, 0), dtype="int64")
    return np.vstack([np.asarray(src, dtype="int64"), np.asarray(dst, dtype="int64")])


if torch is not None:
    class SageLayer(nn.Module):
        def __init__(self, in_dim: int, out_dim: int):
            super().__init__()
            self.self_lin = nn.Linear(in_dim, out_dim)
            self.neigh_lin = nn.Linear(in_dim, out_dim)

        def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
            if edge_index.numel() == 0:
                neigh = torch.zeros_like(x)
            else:
                src, dst = edge_index
                agg = torch.zeros_like(x)
                agg.index_add_(0, dst, x[src])
                deg = torch.bincount(dst, minlength=x.shape[0]).clamp(min=1).to(x.dtype).unsqueeze(1)
                neigh = agg / deg
            return self.self_lin(x) + self.neigh_lin(neigh)


    class GraphSageRegressor(nn.Module):
        def __init__(self, in_dim: int, hidden: int, layers: int, out_dim: int):
            super().__init__()
            mods, norms = [], []
            dim = in_dim
            for _ in range(layers):
                mods.append(SageLayer(dim, hidden))
                norms.append(nn.LayerNorm(hidden))   # normalize each layer
                dim = hidden
            self.layers = nn.ModuleList(mods)
            self.norms = nn.ModuleList(norms)
            self.dropout = nn.Dropout(GNN_DROPOUT)
            self.head = nn.Linear(hidden, out_dim)

        def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
            h = x
            for layer, norm in zip(self.layers, self.norms):
                h = layer(h, edge_index)
                h = norm(h)              # before relu
                h = torch.relu(h)
                h = self.dropout(h)
            return self.head(h)


def _inner_spatial_val_split(df, train_cell_mask, fold, val_frac=0.2, seed=0):
    """Hold out whole training blocks for early stopping."""
    train_blocks = df.loc[train_cell_mask, "spatial_block"].unique()
    rng = np.random.default_rng(seed + fold)
    rng.shuffle(train_blocks)
    n_val = max(1, int(len(train_blocks) * val_frac))
    val_blocks = set(train_blocks[:n_val])
    is_val = df["spatial_block"].isin(val_blocks)
    inner_train_mask = train_cell_mask & (~is_val)
    inner_val_mask = train_cell_mask & is_val
    return inner_train_mask, inner_val_mask


def run_gnn_spatial_cv(df: pd.DataFrame, outcomes: list[str], feature_sets: dict[str, list[str]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    if torch is None:
        print("[SKIP] torch unavailable.")
        return pd.DataFrame(), pd.DataFrame()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    edge_cache = {}
    metric_rows, pred_frames = [], []

    selected_sets = {k: v for k, v in feature_sets.items() if k in GNN_FEATURE_SETS_TO_RUN}
    if not selected_sets:
        print(f"[SKIP] None of GNN_FEATURE_SETS_TO_RUN available: {GNN_FEATURE_SETS_TO_RUN}")
        return pd.DataFrame(), pd.DataFrame()

    # single-task was more stable for weak outcomes
    task_groups = [[o] for o in outcomes] if GNN_SINGLE_TASK else [list(outcomes)]

    def build_edges(h3_arr, fold, tag):
        key = (int(fold), tag, GNN_K_RING)
        if key not in edge_cache:
            edge_cache[key] = build_edge_index(h3_arr, GNN_K_RING)
        return edge_cache[key]

    for set_name, cols in selected_sets.items():
        for group in task_groups:
            tag = "multitask" if len(group) > 1 else group[0]
            print(f"\n=== GNN spatial CV: {set_name} | {tag} ===")
            for fold in range(N_SPATIAL_FOLDS):
                nonbuffer = ~df["is_dhs_boundary_buffer"].fillna(False)
                # separate masks also keep train-test graph edges apart
                train_cell_mask = (df["fold"] != fold) & nonbuffer
                test_cell_mask = (df["fold"] == fold) & nonbuffer

                # inner block validation
                inner_train_mask, inner_val_mask = _inner_spatial_val_split(
                    df, train_cell_mask, fold, val_frac=GNN_VAL_FRAC, seed=RANDOM_STATE
                )

                tr_lbl = df.loc[inner_train_mask, group].notna().any(axis=1).sum()
                va_lbl = df.loc[inner_val_mask, group].notna().any(axis=1).sum()
                te_lbl = df.loc[test_cell_mask, group].notna().any(axis=1).sum()
                if tr_lbl < 20 or te_lbl < 2:
                    print(f"[SKIP] fold={fold} {tag}: train_lbl={tr_lbl}, test_lbl={te_lbl}")
                    continue
                if va_lbl < 10:
                    # sparse fold: use all train cells
                    inner_train_mask, inner_val_mask = train_cell_mask, None

                x_tr_np, spec = fit_matrix(df, inner_train_mask, cols)
                x_te_np = transform_matrix(df, test_cell_mask, cols, spec)
                edge_tr = torch.from_numpy(
                    build_edges(df.loc[inner_train_mask, "h3_index"].astype(str).to_numpy(), fold, f"itrain_{tag}")
                ).to(device)
                edge_te = torch.from_numpy(
                    build_edges(df.loc[test_cell_mask, "h3_index"].astype(str).to_numpy(), fold, "test")
                ).to(device)

                x_tr = torch.from_numpy(x_tr_np).to(device)
                x_te = torch.from_numpy(x_te_np).to(device)
                y_tr_np = df.loc[inner_train_mask, group].astype("float32").to_numpy()
                y_tr = torch.from_numpy(np.nan_to_num(y_tr_np, nan=0.0)).to(device)
                mask_tr = torch.from_numpy(np.isfinite(y_tr_np).astype("float32")).to(device)
                denom_tr = mask_tr.sum().clamp(min=1.0)

                if inner_val_mask is not None:
                    x_va_np = transform_matrix(df, inner_val_mask, cols, spec)
                    edge_va = torch.from_numpy(
                        build_edges(df.loc[inner_val_mask, "h3_index"].astype(str).to_numpy(), fold, f"ival_{tag}")
                    ).to(device)
                    x_va = torch.from_numpy(x_va_np).to(device)
                    y_va_np = df.loc[inner_val_mask, group].astype("float32").to_numpy()
                    y_va = torch.from_numpy(np.nan_to_num(y_va_np, nan=0.0)).to(device)
                    mask_va = torch.from_numpy(np.isfinite(y_va_np).astype("float32")).to(device)
                    denom_va = mask_va.sum().clamp(min=1.0)

                model = GraphSageRegressor(x_tr_np.shape[1], GNN_HIDDEN, GNN_LAYERS, out_dim=len(group)).to(device)
                opt = torch.optim.AdamW(model.parameters(), lr=GNN_LR, weight_decay=GNN_WEIGHT_DECAY)

                best_val = float("inf")
                best_state = None
                patience = GNN_PATIENCE
                bad = 0
                for epoch in range(1, GNN_EPOCHS + 1):
                    model.train()
                    pred = model(x_tr, edge_tr)
                    loss = (((pred - y_tr) ** 2) * mask_tr).sum() / denom_tr
                    opt.zero_grad(); loss.backward(); opt.step()

                    if inner_val_mask is not None:
                        model.eval()
                        with torch.no_grad():
                            vpred = model(x_va, edge_va)
                            vloss = float(((((vpred - y_va) ** 2) * mask_va).sum() / denom_va).cpu())
                        # early stop on validation loss
                        if vloss < best_val - 1e-5:
                            best_val = vloss
                            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                            bad = 0
                        else:
                            bad += 1
                            if bad >= patience:
                                break

                if best_state is not None:
                    model.load_state_dict(best_state)

                model.eval()
                with torch.no_grad():
                    pred_te_all = model(x_te, edge_te).cpu().numpy().astype("float32")

                y_te_all = df.loc[test_cell_mask, group].astype("float32").to_numpy()
                rows_all = df.loc[test_cell_mask].reset_index(drop=True)
                for j, outcome in enumerate(group):
                    ok = np.isfinite(y_te_all[:, j])
                    n_tr_o = int(np.isfinite(y_tr_np[:, j]).sum())
                    if n_tr_o < 20 or int(ok.sum()) < 2:
                        continue
                    y_te = y_te_all[ok, j]; pred = pred_te_all[ok, j]
                    m = regression_metrics(y_te, pred)
                    m.update({
                        "experiment": "spatial_cv",
                        "model": "graphsage_singletask" if GNN_SINGLE_TASK else "strict_multitask_graphsage",
                        "outcome": outcome, "feature_set": set_name, "fold": fold,
                        "n_train": n_tr_o, "n_test": int(ok.sum()),
                        "train_edges": int(edge_tr.shape[1]), "test_edges": int(edge_te.shape[1]),
                        "n_outputs": len(group),
                    })
                    metric_rows.append(m)
                    print(f"fold={fold} {outcome}: r2={m['r2']:.3f} spearman={m['spearman']:.3f}")
                    rows = rows_all.loc[ok]
                    pred_frames.append(pd.DataFrame({
                        "experiment": "spatial_cv", "model": m["model"], "outcome": outcome,
                        "feature_set": set_name, "fold": fold,
                        "country_iso3": rows["country_iso3"].to_numpy(),
                        "h3_index": rows["h3_index"].to_numpy(),
                        "y_true": y_te, "y_pred": pred,
                    }))

    metrics = pd.DataFrame(metric_rows)
    predictions = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    return metrics, predictions


if RUN_GNN:
    gnn_metrics, gnn_predictions = run_gnn_spatial_cv(cells, active_outcomes, feature_sets)
    gnn_metrics_path = TRAINING_OUT / "gnn_spatial_cv_metrics.csv"
    gnn_predictions_path = TRAINING_OUT / "gnn_spatial_cv_predictions.parquet"
    gnn_metrics.to_csv(gnn_metrics_path, index=False)
    gnn_predictions.to_parquet(gnn_predictions_path, compression="snappy", index=False)
    print(f"\nWrote {gnn_metrics_path}")
    print(f"Wrote {gnn_predictions_path}")
    if len(gnn_metrics):
        display(summarize_metrics(gnn_metrics, ["experiment", "model", "outcome", "feature_set"]).round(4))
else:
    gnn_metrics = pd.DataFrame()
    gnn_predictions = pd.DataFrame()
    print("RUN_GNN=False")


=== GNN spatial CV: ae_only | dhs_wealth ===
fold=0 dhs_wealth: r2=0.597 spearman=0.777
fold=1 dhs_wealth: r2=0.589 spearman=0.769
fold=2 dhs_wealth: r2=0.656 spearman=0.822
fold=3 dhs_wealth: r2=0.673 spearman=0.823
fold=4 dhs_wealth: r2=0.694 spearman=0.825

=== GNN spatial CV: ae_only | dhs_sanitation ===
fold=0 dhs_sanitation: r2=0.362 spearman=0.607
fold=1 dhs_sanitation: r2=0.362 spearman=0.620
fold=2 dhs_sanitation: r2=0.456 spearman=0.691
fold=3 dhs_sanitation: r2=0.419 spearman=0.658
fold=4 dhs_sanitation: r2=0.481 spearman=0.717

=== GNN spatial CV: ae_only | dhs_women_edu ===
fold=0 dhs_women_edu: r2=0.669 spearman=0.814
fold=1 dhs_women_edu: r2=0.706 spearman=0.818
fold=2 dhs_women_edu: r2=0.586 spearman=0.786
fold=3 dhs_women_edu: r2=0.682 spearman=0.811
fold=4 dhs_women_edu: r2=0.638 spearman=0.809

=== GNN spatial CV: ae_only | dhs_stunting ===
fold=0 dhs_stunting: r2=0.208 spearman=0.372
fold=1 dhs_stunting: r2=0.213 spearman=0.399
fold=2 dhs_stunting: r2=0.265 spearma

,experiment,model,outcome,feature_set,r2_mean,r2_std,r2_min,r2_max,rmse_mean,rmse_std,rmse_min,rmse_max,mae_mean,mae_std,mae_min,mae_max,spearman_mean,spearman_std,spearman_min,spearman_max,n_mean,n_std,n_min,n_max
0,spatial_cv,graphsage_singletask,dhs_mobile,ae_only,0.3753,0.1170,0.2077,0.5244,0.1174,0.0162,0.0991,0.1369,0.0851,0.0125,0.0713,0.0999,0.5553,0.0585,0.4958,0.6318,853.2,10.6630,836,863
1,spatial_cv,graphsage_singletask,dhs_mobile,full,0.5243,0.0857,0.3963,0.6137,0.1019,0.0059,0.0959,0.1105,0.0725,0.0058,0.0650,0.0806,0.6447,0.0526,0.5765,0.7045,853.2,10.6630,836,863
2,spatial_cv,graphsage_singletask,dhs_sanitation,ae_only,0.4158,0.0540,0.3616,0.4809,0.2591,0.0064,0.2496,0.2675,0.2051,0.0077,0.1953,0.2140,0.6586,0.0463,0.6072,0.7171,853.2,10.6630,836,863
3,spatial_cv,graphsage_singletask,dhs_sanitation,full,0.5094,0.0416,0.4688,0.5659,0.2376,0.0112,0.2229,0.2524,0.1841,0.0074,0.1740,0.1920,0.7232,0.0241,0.6984,0.7513,853.2,10.6630,836,863
4,spatial_cv,graphsage_singletask,dhs_stunting,ae_only,0.2147,0.0347,0.1676,0.2650,0.1946,0.0096,0.1847,0.2089,0.1529,0.0087,0.1436,0.1661,0.4168,0.0455,0.3721,0.4903,843.8,10.8028,829,857
5,spatial_cv,graphsage_singletask,dhs_stunting,full,0.2005,0.0212,0.1671,0.2182,0.1963,0.0054,0.1907,0.2028,0.1537,0.0054,0.1488,0.1603,0.4080,0.0470,0.3467,0.4689,843.8,10.8028,829,857
6,spatial_cv,graphsage_singletask,dhs_wealth,ae_only,0.6419,0.0467,0.5892,0.6943,0.4970,0.0203,0.4748,0.5222,0.3800,0.0170,0.3667,0.4000,0.8031,0.0278,0.7686,0.8253,853.2,10.6630,836,863
7,spatial_cv,graphsage_singletask,dhs_wealth,full,0.7453,0.0472,0.6684,0.7853,0.4181,0.0212,0.3896,0.4418,0.3181,0.0193,0.2937,0.3425,0.8693,0.0311,0.8161,0.8897,853.2,10.6630,836,863
8,spatial_cv,graphsage_singletask,dhs_women_edu,ae_only,0.6563,0.0463,0.5860,0.7057,2.0832,0.1376,1.9054,2.2308,1.5995,0.1078,1.4572,1.7066,0.8075,0.0127,0.7859,0.8183,853.2,10.6630,836,863
9,spatial_cv,graphsage_singletask,dhs_women_edu,full,0.7026,0.0522,0.6353,0.7719,1.9330,0.1406,1.7821,2.0937,1.4733,0.0935,1.3788,1.5721,0.8248,0.0148,0.8074,0.8430,853.2,10.6630,836,863


## Country holdout

All-country run only. held-out country stays out of supervised and SSL fit.

In [44]:
def loco_splits(df: pd.DataFrame):
    countries = sorted(df["country_iso3"].dropna().unique())
    for holdout in countries:
        train_mask = df["country_iso3"] != holdout
        test_mask = df["country_iso3"] == holdout
        yield holdout, train_mask, test_mask


def run_xgb_loco(df: pd.DataFrame, outcomes: list[str], feature_sets: dict[str, list[str]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows = []
    pred_frames = []
    for holdout, train_country_mask, test_country_mask in loco_splits(df):
        for outcome in outcomes:
            train_label_mask = train_country_mask & df[outcome].notna()
            test_label_mask = test_country_mask & df[outcome].notna()
            if int(train_label_mask.sum()) < 20 or int(test_label_mask.sum()) < 2:
                print(f"[SKIP LOCO] {holdout} {outcome}: train={int(train_label_mask.sum())}, test={int(test_label_mask.sum())}")
                continue
            for set_name, cols in feature_sets.items():
                print(f"\n=== XGB LOCO: holdout={holdout} | {outcome} | {set_name} ===")
                x_train, spec = fit_matrix(df, train_label_mask, cols)
                x_test = transform_matrix(df, test_label_mask, cols, spec)
                y_train = df.loc[train_label_mask, outcome].astype("float32").to_numpy()
                y_test = df.loc[test_label_mask, outcome].astype("float32").to_numpy()
                model = make_xgb_or_fallback()
                try:
                    model.fit(x_train, y_train)
                except Exception as exc:
                    if xgb is not None and "cuda" in str(exc).lower():
                        print(f"[WARN] CUDA XGBoost failed for LOCO holdout={holdout}; retrying CPU hist. Error: {exc}")
                        params = dict(XGB_PARAMS)
                        model = xgb.XGBRegressor(**params)
                        model.fit(x_train, y_train)
                    else:
                        raise
                pred = model.predict(x_test).astype("float32")
                m = regression_metrics(y_test, pred)
                m.update({
                    "experiment": "loco",
                    "model": "xgb" if xgb is not None else "hist_gradient_boosting",
                    "outcome": outcome,
                    "feature_set": set_name,
                    "holdout_country": holdout,
                    "n_train": int(train_label_mask.sum()),
                    "n_test": int(test_label_mask.sum()),
                })
                metric_rows.append(m)
                print(f"holdout={holdout} r2={m['r2']:.3f} rmse={m['rmse']:.3f} spearman={m['spearman']:.3f}")
                pred_frames.append(pd.DataFrame({
                    "experiment": "loco",
                    "model": m["model"],
                    "outcome": outcome,
                    "feature_set": set_name,
                    "holdout_country": holdout,
                    "country_iso3": df.loc[test_label_mask, "country_iso3"].to_numpy(),
                    "h3_index": df.loc[test_label_mask, "h3_index"].to_numpy(),
                    "y_true": y_test,
                    "y_pred": pred,
                }))
    metrics = pd.DataFrame(metric_rows)
    predictions = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    return metrics, predictions


def run_ssl_loco(df: pd.DataFrame, outcomes: list[str], full_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    if torch is None:
        return pd.DataFrame(), pd.DataFrame()
    metric_rows = []
    pred_frames = []
    for holdout, train_country_mask, test_country_mask in loco_splits(df):
        print(f"\n=== SSL LOCO encoder: holdout={holdout} ===")
        x_ssl_train, spec = fit_matrix(df, train_country_mask, full_cols)
        model, ae_idx, tab_idx = train_ssl_encoder(x_ssl_train, spec)
        for outcome in outcomes:
            train_label_mask = train_country_mask & df[outcome].notna()
            test_label_mask = test_country_mask & df[outcome].notna()
            if int(train_label_mask.sum()) < 20 or int(test_label_mask.sum()) < 2:
                print(f"[SKIP SSL LOCO] {holdout} {outcome}: train={int(train_label_mask.sum())}, test={int(test_label_mask.sum())}")
                continue
            x_probe_train = transform_matrix(df, train_label_mask, full_cols, spec)
            x_probe_test = transform_matrix(df, test_label_mask, full_cols, spec)
            z_train = encode_with_ssl(model, x_probe_train, ae_idx, tab_idx)
            z_test = encode_with_ssl(model, x_probe_test, ae_idx, tab_idx)
            y_train = df.loc[train_label_mask, outcome].astype("float32").to_numpy()
            y_test = df.loc[test_label_mask, outcome].astype("float32").to_numpy()
            probe = RidgeCV(alphas=np.logspace(-3, 3, 13))
            probe.fit(z_train, y_train)
            pred = probe.predict(z_test).astype("float32")
            m = regression_metrics(y_test, pred)
            m.update({
                "experiment": "loco",
                "model": "ssl_frozen_z_ridge",
                "outcome": outcome,
                "feature_set": "ssl_full_z",
                "holdout_country": holdout,
                "n_train": int(train_label_mask.sum()),
                "n_test": int(test_label_mask.sum()),
                "z_dim": SSL_Z_DIM,
            })
            metric_rows.append(m)
            print(f"holdout={holdout} {outcome}: r2={m['r2']:.3f} rmse={m['rmse']:.3f} spearman={m['spearman']:.3f}")
            pred_frames.append(pd.DataFrame({
                "experiment": "loco",
                "model": "ssl_frozen_z_ridge",
                "outcome": outcome,
                "feature_set": "ssl_full_z",
                "holdout_country": holdout,
                "country_iso3": df.loc[test_label_mask, "country_iso3"].to_numpy(),
                "h3_index": df.loc[test_label_mask, "h3_index"].to_numpy(),
                "y_true": y_test,
                "y_pred": pred,
            }))
    metrics = pd.DataFrame(metric_rows)
    predictions = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    return metrics, predictions


if RUN_LOCO:
    if cells["country_iso3"].nunique() < 3:
        print("[SKIP LOCO] Fewer than 3 countries loaded.")
        loco_xgb_metrics = pd.DataFrame()
        loco_xgb_predictions = pd.DataFrame()
        loco_ssl_metrics = pd.DataFrame()
        loco_ssl_predictions = pd.DataFrame()
    else:
        loco_xgb_metrics, loco_xgb_predictions = run_xgb_loco(cells, active_outcomes, feature_sets)
        loco_xgb_metrics.to_csv(TRAINING_OUT / "xgb_loco_metrics.csv", index=False)
        loco_xgb_predictions.to_parquet(TRAINING_OUT / "xgb_loco_predictions.parquet", compression="snappy", index=False)

        if RUN_SSL_EMBEDDING and "full" in feature_sets:
            loco_ssl_metrics, loco_ssl_predictions = run_ssl_loco(cells, active_outcomes, feature_sets["full"])
            loco_ssl_metrics.to_csv(TRAINING_OUT / "ssl_frozen_loco_metrics.csv", index=False)
            loco_ssl_predictions.to_parquet(TRAINING_OUT / "ssl_frozen_loco_predictions.parquet", compression="snappy", index=False)
        else:
            loco_ssl_metrics = pd.DataFrame()
            loco_ssl_predictions = pd.DataFrame()
        display(summarize_metrics(loco_xgb_metrics, ["experiment", "model", "outcome", "feature_set"]).round(4))
        if len(loco_ssl_metrics):
            display(summarize_metrics(loco_ssl_metrics, ["experiment", "model", "outcome", "feature_set"]).round(4))
else:
    print("RUN_LOCO=False. This is expected when SUBSET_NGA_TZA_ONLY=True.")


=== XGB LOCO: holdout=GHA | dhs_wealth | ae_only ===
holdout=GHA r2=0.564 rmse=0.538 spearman=0.791

=== XGB LOCO: holdout=GHA | dhs_wealth | tab_only ===
holdout=GHA r2=0.533 rmse=0.557 spearman=0.830

=== XGB LOCO: holdout=GHA | dhs_wealth | full ===
holdout=GHA r2=0.634 rmse=0.493 spearman=0.836

=== XGB LOCO: holdout=GHA | dhs_wealth | ae_plus_orthogonal ===
holdout=GHA r2=0.644 rmse=0.486 spearman=0.821

=== XGB LOCO: holdout=GHA | dhs_wealth | ae_plus_amount ===
holdout=GHA r2=0.641 rmse=0.488 spearman=0.830

=== XGB LOCO: holdout=GHA | dhs_sanitation | ae_only ===
holdout=GHA r2=0.280 rmse=0.281 spearman=0.623

=== XGB LOCO: holdout=GHA | dhs_sanitation | tab_only ===
holdout=GHA r2=0.131 rmse=0.309 spearman=0.569

=== XGB LOCO: holdout=GHA | dhs_sanitation | full ===
holdout=GHA r2=0.215 rmse=0.293 spearman=0.562

=== XGB LOCO: holdout=GHA | dhs_sanitation | ae_plus_orthogonal ===
holdout=GHA r2=0.278 rmse=0.281 spearman=0.580

=== XGB LOCO: holdout=GHA | dhs_sanitation | ae_p

,experiment,model,outcome,feature_set,r2_mean,r2_std,r2_min,r2_max,rmse_mean,rmse_std,rmse_min,rmse_max,mae_mean,mae_std,mae_min,mae_max,spearman_mean,spearman_std,spearman_min,spearman_max,n_mean,n_std,n_min,n_max
0,loco,xgb,dhs_mobile,ae_only,-2.4523,5.9329,-15.8699,0.2434,0.1521,0.0825,0.0882,0.3286,0.1204,0.0671,0.0703,0.2571,0.3788,0.1790,0.0371,0.5561,740.8571,487.3669,172,1490
1,loco,xgb,dhs_mobile,ae_plus_amount,-2.2974,5.9461,-15.7518,0.2702,0.1451,0.0822,0.0748,0.3185,0.1128,0.0668,0.0564,0.2482,0.4399,0.1766,0.0946,0.6168,740.8571,487.3669,172,1490
2,loco,xgb,dhs_mobile,ae_plus_orthogonal,-1.7359,4.3548,-11.5651,0.2678,0.1440,0.0833,0.0781,0.3255,0.1119,0.0670,0.0583,0.2535,0.4228,0.1608,0.1077,0.5892,740.8571,487.3669,172,1490
3,loco,xgb,dhs_mobile,full,-2.2242,5.8115,-15.3742,0.2861,0.1435,0.0813,0.0739,0.3149,0.1112,0.0661,0.0556,0.2443,0.4430,0.1730,0.1033,0.6153,740.8571,487.3669,172,1490
4,loco,xgb,dhs_mobile,tab_only,-1.3366,3.1523,-8.4296,0.2611,0.1418,0.0758,0.0828,0.3095,0.1072,0.0602,0.0593,0.2386,0.4311,0.1852,0.0534,0.6009,740.8571,487.3669,172,1490
5,loco,xgb,dhs_sanitation,ae_only,0.2351,0.1782,-0.1156,0.3840,0.2849,0.0349,0.2487,0.3542,0.2327,0.0321,0.1974,0.2920,0.5476,0.1144,0.3822,0.6754,740.8571,487.3669,172,1490
6,loco,xgb,dhs_sanitation,ae_plus_amount,0.3653,0.1154,0.2135,0.5295,0.2596,0.0234,0.2369,0.2936,0.2073,0.0223,0.1871,0.2353,0.6048,0.1075,0.4617,0.7520,740.8571,487.3669,172,1490
7,loco,xgb,dhs_sanitation,ae_plus_orthogonal,0.3339,0.1147,0.1883,0.4679,0.2662,0.0241,0.2403,0.3021,0.2133,0.0261,0.1869,0.2517,0.5869,0.1053,0.4481,0.7208,740.8571,487.3669,172,1490
8,loco,xgb,dhs_sanitation,full,0.3640,0.1113,0.2147,0.5246,0.2601,0.0245,0.2381,0.2934,0.2084,0.0242,0.1873,0.2380,0.6116,0.0958,0.4653,0.7516,740.8571,487.3669,172,1490
9,loco,xgb,dhs_sanitation,tab_only,0.3069,0.1892,-0.0222,0.5207,0.2694,0.0259,0.2391,0.3086,0.2135,0.0201,0.1877,0.2384,0.6042,0.0822,0.4772,0.7210,740.8571,487.3669,172,1490


,experiment,model,outcome,feature_set,r2_mean,r2_std,r2_min,r2_max,rmse_mean,rmse_std,rmse_min,rmse_max,mae_mean,mae_std,mae_min,mae_max,spearman_mean,spearman_std,spearman_min,spearman_max,n_mean,n_std,n_min,n_max
0,loco,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,-1.7713,3.0043,-8.2275,0.3538,0.1568,0.0763,0.1224,0.3296,0.1218,0.0610,0.0837,0.2588,0.3787,0.1884,0.0525,0.6477,740.8571,487.3669,172,1490
1,loco,ssl_frozen_z_ridge,dhs_sanitation,ssl_full_z,0.3386,0.1154,0.1814,0.4860,0.2651,0.0216,0.2405,0.2996,0.2140,0.0174,0.1925,0.2368,0.6022,0.0787,0.5278,0.7124,740.8571,487.3669,172,1490
2,loco,ssl_frozen_z_ridge,dhs_stunting,ssl_full_z,-0.4590,0.6365,-1.7659,0.0234,0.2194,0.0351,0.1812,0.2900,0.1774,0.0255,0.1538,0.2327,0.2398,0.0798,0.1066,0.3050,733.1429,480.7822,172,1489
3,loco,ssl_frozen_z_ridge,dhs_wealth,ssl_full_z,0.6261,0.0748,0.5308,0.7177,0.4903,0.0455,0.4440,0.5586,0.3797,0.0494,0.3158,0.4497,0.7501,0.0919,0.5536,0.8484,740.8571,487.3669,172,1490
4,loco,ssl_frozen_z_ridge,dhs_women_edu,ssl_full_z,-0.4014,1.4978,-3.7772,0.4267,2.9269,1.2057,1.7024,5.1133,2.5088,1.2153,1.3769,4.7335,0.6054,0.0957,0.4360,0.7000,740.8571,487.3669,172,1490


In [45]:
# label counts by country
for outcome in OUTCOME_COLS:
    print(f"\n{outcome}:")
    labeled = cells[cells[outcome].notna()]
    print(labeled.groupby("country_iso3").size().to_string())
    print(f"  TOTAL labeled: {len(labeled)}")
    # after boundary buffer
    nonbuf = labeled[~labeled["is_dhs_boundary_buffer"].fillna(False)]
    print(f"  after buffer:  {len(nonbuf)}")


dhs_wealth:
country_iso3
GHA     602
GMB     172
KEN    1490
NGA    1350
SLE     464
TZA     596
ZMB     512
  TOTAL labeled: 5186
  after buffer:  4266

dhs_sanitation:
country_iso3
GHA     602
GMB     172
KEN    1490
NGA    1350
SLE     464
TZA     596
ZMB     512
  TOTAL labeled: 5186
  after buffer:  4266

dhs_women_edu:
country_iso3
GHA     602
GMB     172
KEN    1490
NGA    1350
SLE     464
TZA     596
ZMB     512
  TOTAL labeled: 5186
  after buffer:  4266

dhs_stunting:
country_iso3
GHA     600
GMB     172
KEN    1489
NGA    1313
SLE     454
TZA     592
ZMB     512
  TOTAL labeled: 5132
  after buffer:  4219

dhs_mobile:
country_iso3
GHA     602
GMB     172
KEN    1490
NGA    1350
SLE     464
TZA     596
ZMB     512
  TOTAL labeled: 5186
  after buffer:  4266


## Results

Metrics and prediction tables from completed runs.

In [50]:
all_metric_frames = []
for name in ["xgb_metrics", "ssl_metrics", "gnn_metrics", "loco_xgb_metrics", "loco_ssl_metrics"]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and len(obj):
        all_metric_frames.append(obj.copy())

all_prediction_frames = []
for name in ["xgb_predictions", "ssl_predictions", "gnn_predictions", "loco_xgb_predictions", "loco_ssl_predictions"]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and len(obj):
        all_prediction_frames.append(obj.copy())

if all_metric_frames:
    all_metrics = pd.concat(all_metric_frames, ignore_index=True, sort=False)
    all_metrics_path = TRAINING_OUT / "all_metrics.csv"
    all_metrics.to_csv(all_metrics_path, index=False)
    print(f"Wrote combined metrics -> {all_metrics_path}")

    spatial = all_metrics[all_metrics["experiment"] == "spatial_cv"].copy()
    if len(spatial):
        print("\nSpatial CV mean +/- std:")
        summary = summarize_metrics(spatial, ["model", "outcome", "feature_set"])
        display(summary.round(4))

    loco = all_metrics[all_metrics["experiment"] == "loco"].copy()
    if len(loco):
        print("\nLOCO per held-out country:")
        display(loco.sort_values(["outcome", "model", "feature_set", "holdout_country"]).round(4))
else:
    print("[WARN] No metrics were produced. Check run switches and label counts.")

if all_prediction_frames:
    all_predictions = pd.concat(all_prediction_frames, ignore_index=True, sort=False)
    pooled_metrics = pooled_metrics_from_predictions(
        all_predictions,
        ["experiment", "model", "outcome", "feature_set"],
    )
    pooled_metrics_path = TRAINING_OUT / "pooled_prediction_metrics.csv"
    pooled_metrics.to_csv(pooled_metrics_path, index=False)
    print(f"\nWrote pooled prediction metrics -> {pooled_metrics_path}")

    spatial_pooled = pooled_metrics[pooled_metrics["experiment"] == "spatial_cv"].copy()
    if len(spatial_pooled):
        print("\nSpatial CV pooled metrics across all held-out fold predictions:")
        display(spatial_pooled.sort_values(["outcome", "model", "feature_set"]).round(4))

    loco_pooled = pooled_metrics[pooled_metrics["experiment"] == "loco"].copy()
    if len(loco_pooled):
        print("\nLOCO pooled metrics across held-out countries:")
        display(loco_pooled.sort_values(["outcome", "model", "feature_set"]).round(4))
else:
    print("[WARN] No prediction tables were produced, so pooled metrics cannot be computed.")

print(f"\nTraining output folder:\n{TRAINING_OUT}")

Wrote combined metrics -> h3_tabular_v2/training_outputs/strict_spatial_20260616_013629/all_metrics.csv

Spatial CV mean +/- std:


,model,outcome,feature_set,r2_mean,r2_std,r2_min,r2_max,rmse_mean,rmse_std,rmse_min,rmse_max,mae_mean,mae_std,mae_min,mae_max,spearman_mean,spearman_std,spearman_min,spearman_max,n_mean,n_std,n_min,n_max
0,graphsage_singletask,dhs_mobile,ae_only,0.3753,0.1170,0.2077,0.5244,0.1174,0.0162,0.0991,0.1369,0.0851,0.0125,0.0713,0.0999,0.5553,0.0585,0.4958,0.6318,853.2,10.6630,836,863
1,graphsage_singletask,dhs_mobile,full,0.5243,0.0857,0.3963,0.6137,0.1019,0.0059,0.0959,0.1105,0.0725,0.0058,0.0650,0.0806,0.6447,0.0526,0.5765,0.7045,853.2,10.6630,836,863
2,graphsage_singletask,dhs_sanitation,ae_only,0.4158,0.0540,0.3616,0.4809,0.2591,0.0064,0.2496,0.2675,0.2051,0.0077,0.1953,0.2140,0.6586,0.0463,0.6072,0.7171,853.2,10.6630,836,863
3,graphsage_singletask,dhs_sanitation,full,0.5094,0.0416,0.4688,0.5659,0.2376,0.0112,0.2229,0.2524,0.1841,0.0074,0.1740,0.1920,0.7232,0.0241,0.6984,0.7513,853.2,10.6630,836,863
4,graphsage_singletask,dhs_stunting,ae_only,0.2147,0.0347,0.1676,0.2650,0.1946,0.0096,0.1847,0.2089,0.1529,0.0087,0.1436,0.1661,0.4168,0.0455,0.3721,0.4903,843.8,10.8028,829,857
5,graphsage_singletask,dhs_stunting,full,0.2005,0.0212,0.1671,0.2182,0.1963,0.0054,0.1907,0.2028,0.1537,0.0054,0.1488,0.1603,0.4080,0.0470,0.3467,0.4689,843.8,10.8028,829,857
6,graphsage_singletask,dhs_wealth,ae_only,0.6419,0.0467,0.5892,0.6943,0.4970,0.0203,0.4748,0.5222,0.3800,0.0170,0.3667,0.4000,0.8031,0.0278,0.7686,0.8253,853.2,10.6630,836,863
7,graphsage_singletask,dhs_wealth,full,0.7453,0.0472,0.6684,0.7853,0.4181,0.0212,0.3896,0.4418,0.3181,0.0193,0.2937,0.3425,0.8693,0.0311,0.8161,0.8897,853.2,10.6630,836,863
8,graphsage_singletask,dhs_women_edu,ae_only,0.6563,0.0463,0.5860,0.7057,2.0832,0.1376,1.9054,2.2308,1.5995,0.1078,1.4572,1.7066,0.8075,0.0127,0.7859,0.8183,853.2,10.6630,836,863
9,graphsage_singletask,dhs_women_edu,full,0.7026,0.0522,0.6353,0.7719,1.9330,0.1406,1.7821,2.0937,1.4733,0.0935,1.3788,1.5721,0.8248,0.0148,0.8074,0.8430,853.2,10.6630,836,863



LOCO per held-out country:


,n,r2,rmse,mae,spearman,experiment,model,outcome,feature_set,fold,n_train,n_test,z_dim,train_edges,test_edges,n_outputs,holdout_country
379,602,-2.4874,0.1346,0.1016,0.4477,loco,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,NaN,4584,602,256.0,NaN,NaN,NaN,GHA
384,172,-8.2275,0.1265,0.1101,0.0525,loco,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,NaN,5014,172,256.0,NaN,NaN,NaN,GMB
389,1490,-1.0254,0.1303,0.1058,0.3057,loco,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,NaN,3696,1490,256.0,NaN,NaN,NaN,KEN
394,1350,-0.0552,0.1224,0.0837,0.4257,loco,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,NaN,3836,1350,256.0,NaN,NaN,NaN,NGA
399,464,-1.0273,0.3296,0.2588,0.2810,loco,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,NaN,4722,464,256.0,NaN,NaN,NaN,SLE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,1490,0.3063,2.5630,2.1575,0.6238,loco,xgb,dhs_women_edu,tab_only,NaN,3696,1490,NaN,NaN,NaN,NaN,KEN
286,1350,0.4185,3.4270,3.0426,0.7970,loco,xgb,dhs_women_edu,tab_only,NaN,3836,1350,NaN,NaN,NaN,NaN,NGA
311,464,-2.8402,4.5845,4.2132,0.5258,loco,xgb,dhs_women_edu,tab_only,NaN,4722,464,NaN,NaN,NaN,NaN,SLE
336,596,0.2977,1.8030,1.4276,0.6444,loco,xgb,dhs_women_edu,tab_only,NaN,4590,596,NaN,NaN,NaN,NaN,TZA



Wrote pooled prediction metrics -> h3_tabular_v2/training_outputs/strict_spatial_20260616_013629/pooled_prediction_metrics.csv

Spatial CV pooled metrics across all held-out fold predictions:


,experiment,model,outcome,feature_set,n,r2,rmse,mae,spearman,n_folds,n_holdout_countries
30,spatial_cv,graphsage_singletask,dhs_mobile,ae_only,4266,0.3844,0.1182,0.0851,0.5462,5,0
31,spatial_cv,graphsage_singletask,dhs_mobile,full,4266,0.5414,0.1020,0.0725,0.6406,5,0
40,spatial_cv,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,4266,0.3179,0.1244,0.0845,0.5988,5,0
45,spatial_cv,xgb,dhs_mobile,ae_only,4266,0.4676,0.1099,0.0761,0.6281,5,0
46,spatial_cv,xgb,dhs_mobile,ae_plus_amount,4266,0.5043,0.1061,0.0727,0.6634,5,0
47,spatial_cv,xgb,dhs_mobile,ae_plus_orthogonal,4266,0.4976,0.1068,0.0732,0.6562,5,0
48,spatial_cv,xgb,dhs_mobile,full,4266,0.5158,0.1048,0.0721,0.6666,5,0
49,spatial_cv,xgb,dhs_mobile,tab_only,4266,0.5009,0.1064,0.0739,0.6521,5,0
32,spatial_cv,graphsage_singletask,dhs_sanitation,ae_only,4266,0.4215,0.2591,0.2051,0.6627,5,0
33,spatial_cv,graphsage_singletask,dhs_sanitation,full,4266,0.5128,0.2378,0.1840,0.7265,5,0



LOCO pooled metrics across held-out countries:


,experiment,model,outcome,feature_set,n,r2,rmse,mae,spearman,n_folds,n_holdout_countries
0,loco,ssl_frozen_z_ridge,dhs_mobile,ssl_full_z,5186,-0.1188,0.1569,0.1114,0.3166,0,7
5,loco,xgb,dhs_mobile,ae_only,5186,0.0038,0.1480,0.1012,0.3335,0,7
6,loco,xgb,dhs_mobile,ae_plus_amount,5186,0.1057,0.1403,0.0937,0.4524,0,7
7,loco,xgb,dhs_mobile,ae_plus_orthogonal,5186,0.0845,0.1419,0.0945,0.4287,0,7
8,loco,xgb,dhs_mobile,full,5186,0.1261,0.1386,0.0919,0.4784,0,7
9,loco,xgb,dhs_mobile,tab_only,5186,0.0973,0.1409,0.0947,0.4763,0,7
1,loco,ssl_frozen_z_ridge,dhs_sanitation,ssl_full_z,5186,0.3803,0.2688,0.2164,0.6366,0,7
10,loco,xgb,dhs_sanitation,ae_only,5186,0.2806,0.2896,0.2386,0.5461,0,7
11,loco,xgb,dhs_sanitation,ae_plus_amount,5186,0.4147,0.2612,0.2094,0.6519,0,7
12,loco,xgb,dhs_sanitation,ae_plus_orthogonal,5186,0.3873,0.2672,0.2155,0.6315,0,7



Training output folder:
h3_tabular_v2/training_outputs/strict_spatial_20260616_013629
